# QwenASR on Colab（無頭版）
把 Qwen3-ASR-1.7B 跑在 Colab GPU 上做語音轉字幕，精度 fp16／bf16（非 int 量化）。

**流程**：參數 → 安裝 → 檢查 GPU → 載入模型 → 上傳音檔 → 轉錄 → 下載 SRT

> 免費版 Colab：T4（約 15GB VRAM），session 上限約 12 小時，閒置會斷線。
> 斷線後重跑整份 notebook 即可（模型要重下，主模型約 4.7GB，幾分鐘）。

In [ ]:
# ═══ 參數（跑之前先看這格） ═══

# 模型：用無 -hf 的原版 repo（權重鍵為 thinker.*，對應 qwen-asr 內建模型類別）。
# 注意：-hf 版是 transformers 原生重發版（鍵為 model.*），qwen-asr 0.0.6 讀它會鍵名不匹配。
# 日語動漫特化版改用下面那行。
MODEL_ID = "Qwen/Qwen3-ASR-1.7B"
# MODEL_ID = "jaykwok/Qwen3-ASR-1.7B-JA-Anime-Galgame-hf"  # 日語動漫版

# 字級時間軸對齊模型（卡拉OK逐字、精準字幕）。關掉可省 ~1.2GB 下載。
ALIGNER_ID = "Qwen/Qwen3-ForcedAligner-0.6B"
USE_ALIGNER = True

# 語言：已知語言時指定較準；None = 自動偵測（預設）
# 可選 "Chinese" / "Japanese" / "English" / "Cantonese" ...（30 種）
LANGUAGE = None

# 辨識提示（人名、術語、背景說明），可留空
HINT = ""

# 簡轉繁：中文才轉，日語自動跳過
DO_OPENCC = True  # True=s2twp（含台灣用語）


In [ ]:
# ═══ 安裝依賴 ═══
# Colab 已內建 torch+CUDA，只需補 ASR 相關套件
# 注意：qwen-asr 官方鎖定 transformers==4.57.6（5.x 會壞），不要加 -U transformers
!pip install -q "transformers==4.57.6" qwen-asr accelerate safetensors onnxruntime soundfile opencc-python-reimplemented huggingface_hub tqdm


In [ ]:
# ═══ GPU 檢查 ═══
import torch, sys
assert torch.cuda.is_available(), "❌ 沒有偵測到 GPU！請到「執行階段 → 變更執行階段類型」選 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} / VRAM: {vram:.1f} GB / torch {torch.__version__} / cuda {torch.version.cuda}")

# 精度決策：T4（Turing）無原生 bf16 → 用 fp16；A100/L4 等才用 bf16
DTYPE = torch.float16 if "T4" in name else torch.bfloat16
print(f"使用精度：{str(DTYPE).split(chr(46))[-1]}")


In [ ]:
# ═══ 下載 VAD 模型 + 載入 ASR 模型 ═══
import os, urllib.request, time
VAD_PATH = "/content/silero_vad.onnx"
if not os.path.exists(VAD_PATH):
    print("下載 Silero VAD ...")
    urllib.request.urlretrieve("https://github.com/snakers4/silero-vad/raw/v4.0/files/silero_vad.onnx", VAD_PATH)
print("VAD OK")

import onnxruntime as ort
vad_sess = ort.InferenceSession(VAD_PATH, providers=["CPUExecutionProvider"])

from qwen_asr import Qwen3ASRModel

# ── qwen-asr 0.0.6 上游 bug 的 workaround ─────────────────────────────
# vendored 的 Qwen3ASRThinkerTextRotaryEmbedding 在 rope_scaling=None
# （官方 config 就是 None）時會 AttributeError 炸掉，所有版本都有此 bug。
# rope_type=default 時 mrope_section 根本用不到，補預設值即可。
import qwen_asr.core.transformers_backend.modeling_qwen3_asr as _M
_orig_rope_init = _M.Qwen3ASRThinkerTextRotaryEmbedding.__init__
def _fixed_rope_init(self, config, device=None):
    if getattr(config, 'rope_scaling', None) is None:
        config.rope_scaling = {'rope_type': 'default', 'mrope_section': [24, 20, 20]}
    _orig_rope_init(self, config, device)
_M.Qwen3ASRThinkerTextRotaryEmbedding.__init__ = _fixed_rope_init
# 注意：_orig_rope_init 不可 del，_fixed_rope_init 執行時要從 global 找它

# ── 主模型：streaming 載入 ──────────────────────────────────────────
# free Colab 只有 12.7GB 系統 RAM。transformers 的 from_pretrained 預設會在
# RAM 留雙份（整片 3.8GB 權重＋CPU 上的模型本體），且 low_cpu_mem_usage 對
# from_pretrained 會在 RAM staging 權重（12.67GB 不夠，已實測 OOM 三次）。
# 改用 meta 空殼初始化＋ safetensors 逐分片載入直上 GPU：v4/v5 實測 RAM 峰值約 5.4GB。
import torch, glob
from huggingface_hub import snapshot_download
from transformers import AutoProcessor
from accelerate import init_empty_weights
from accelerate.utils import set_module_tensor_to_device
from safetensors import safe_open
# 直接用 vendored 的正確類別實例化（與 qwen-asr 內部註冊的映射一致，
# 不走 AutoModel 自動映射，避免解析到錯誤類別）
from qwen_asr.core.transformers_backend.configuration_qwen3_asr import Qwen3ASRConfig
from qwen_asr.core.transformers_backend.modeling_qwen3_asr import Qwen3ASRForConditionalGeneration

t0 = time.time()
print(f"下載 {MODEL_ID} ...（首次約 4GB，之後用快取）")
snap = snapshot_download(MODEL_ID)
ckpts = sorted(glob.glob(os.path.join(snap, '*.safetensors')))
print('權重分片：', [os.path.basename(c) for c in ckpts])
config = Qwen3ASRConfig.from_pretrained(MODEL_ID)
with init_empty_weights():
    hf_model = Qwen3ASRForConditionalGeneration(config)
print('逐 tensor 載入權重到 GPU ...')
# meta 模型不能用 load_state_dict／load_model：它們只把值 copy 進現有 storage，
# 而 meta tensor 沒有 storage（v6/v7 實測 708 個參數全殘留 meta）。
# 正確做法：用 accelerate 的 set_module_tensor_to_device 逐 tensor 取代參數物件
#（這也是 accelerate 自己載入檢查點用的機制）。
_expected = set(hf_model.state_dict().keys())
for ckpt in ckpts:
    print(' ', os.path.basename(ckpt), flush=True)
    with safe_open(ckpt, framework='pt') as _f:
        for _wn in _f.keys():
            assert _wn in _expected, f'鍵名不匹配: {_wn}'
            _t = _f.get_tensor(_wn).to(DTYPE)  # 官方權重是 bf16，這裡直接轉目標精度
            set_module_tensor_to_device(hf_model, _wn, 'cuda:0', value=_t)
            del _t
del ckpt, ckpts, _expected
_meta_left = [n for n, _p in hf_model.named_parameters() if _p.is_meta]
assert not _meta_left, f'有權重沒載到: {_meta_left[:5]}'
del _meta_left
# non-persistent buffers（如 SinusoidsPositionEmbedding、rotary inv_freq）在
# init_empty_weights 下預設留在 CPU，前向傳播時跟 CUDA tensor 相加會炸 → 搬上 cuda 並轉精度
for _n, _b in hf_model.named_buffers():
    if _b.device.type != 'cuda':
        _b.data = _b.data.to('cuda:0', dtype=DTYPE)
del _n, _b
# 最終門：所有參數與 buffer 都要在 cuda 上、無 meta 殘留
_bad = [n for n, _p in hf_model.named_parameters() if _p.is_meta or _p.device.type != 'cuda']
_bad += [n for n, _b in hf_model.named_buffers() if _b.is_meta or _b.device.type != 'cuda']
assert not _bad, f'裝置不對: {_bad[:5]}'
del _bad
hf_model.eval()
torch.cuda.empty_cache()
# 跟 qwen-asr 自己的寫法一致：不傳 trust_remote_code，走內建註冊的 processor
processor = AutoProcessor.from_pretrained(MODEL_ID, fix_mistral_regex=True)
model = Qwen3ASRModel(
    backend='transformers',
    model=hf_model,
    processor=processor,
    sampling_params=None,
    forced_aligner=None,
    max_inference_batch_size=32,
    max_new_tokens=512,
)
print(f"✅ 主模型就緒（耗時 {time.time()-t0:.0f}s）")
import gc; gc.collect(); torch.cuda.empty_cache()
if USE_ALIGNER:
    try:
        from qwen_asr import Qwen3ForcedAligner
        t1 = time.time()
        print(f"載入對齊模型 {ALIGNER_ID} ...")
        model.forced_aligner = Qwen3ForcedAligner.from_pretrained(
            ALIGNER_ID, device_map="cuda", dtype=DTYPE, low_cpu_mem_usage=True,
        )
        print(f"✅ 對齊模型就緒（耗時 {time.time()-t1:.0f}s）")
    except Exception as _e:
        model.forced_aligner = None
        print(f"⚠ 對齊模型載入失敗，字級時間軸改用比例估算：{type(_e).__name__}: {_e}")
# 以實際載入結果為準：沒載起來就自動降級，轉錄格會走 fallback
USE_ALIGNER = bool(USE_ALIGNER and model.forced_aligner is not None)
print("VRAM 已用：%.1f GB" % (torch.cuda.memory_allocated(0)/1024**3))


In [ ]:
# ═══ 上傳音檔 ═══
# 執行後會跳出檔案選擇器（mp3/wav/flac/m4a/ogg/mp4/mkv 皆可，影片會自動抽音軌）。
# 若 /content 已有音檔（例如先用 wget 抓下來），會直接使用，不跳選擇器。
import os
_AUDIO_EXTS = ('.mp3', '.wav', '.flac', '.m4a', '.ogg', '.opus',
                '.mp4', '.mkv', '.mov', '.avi', '.wma', '.aac')
_existing = sorted(f for f in os.listdir('/content')
                  if f.lower().endswith(_AUDIO_EXTS)
                  and os.path.isfile('/content/' + f))
if _existing:
    AUDIO_PATH = '/content/' + _existing[0]
    print("使用 /content 現有音檔：", AUDIO_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    AUDIO_PATH = "/content/" + list(uploaded.keys())[0]
    print("輸入：", AUDIO_PATH)


In [ ]:
# ═══ （可選）快速測試：下載一段 4 秒中文範例音檔 ═══
# 只是想驗證環境有沒有跑起來時執行這格，再跑下面的轉錄格即可。
# !wget -q https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen3-ASR-Repo/asr_zh.wav -O /content/test_zh.wav && echo OK


In [ ]:
# ═══ 核心函式（VAD 切段 / 轉錄 / SRT） ═══
# 邏輯移植自 QwenASRMiniTool 的 app-gpu.py（已在 Windows+CUDA 驗證過）
import subprocess, shutil
import numpy as np

SR = 16000
VAD_CHUNK = 512
VAD_THRESHOLD = 0.5
MAX_GROUP_SEC = 20
MAX_CHARS = 20
MIN_SUB_SEC = 0.6

_ZH_CLAUSE_END = frozenset('，。？！；：…—、·')
_EN_SENT_END = frozenset('.,!?;')

def load_audio_16k_mono(path):
    """讀成 float32 mono @16k。soundfile 先讀，讀不了就用 ffmpeg 轉。"""
    import soundfile as sf
    try:
        data, sr = sf.read(path, always_2d=False)
    except Exception:
        ff = shutil.which('ffmpeg')
        assert ff, '此格式需要 ffmpeg，但環境中找不到'
        tmp = '/tmp/_asr_in.wav'
        subprocess.run([ff, '-y', '-v', 'error', '-i', path,
                        '-ac', '1', '-ar', '16000', '-f', 'wav', tmp], check=True)
        data, sr = sf.read(tmp, always_2d=False)
    data = np.asarray(data, dtype=np.float32)
    if data.ndim > 1:
        data = data.mean(axis=1)
    if sr != SR:
        n_out = int(len(data) * SR / sr)
        data = np.interp(np.linspace(0, len(data)-1, n_out),
                         np.arange(len(data)), data).astype(np.float32)
    peak = np.max(np.abs(data))
    if peak > 1.0:
        data = data / peak
    return data

def detect_speech_groups(audio, vad_sess):
    """Silero VAD 切段 → [(start_s, end_s, chunk), ...]"""
    h = np.zeros((2, 1, 64), dtype=np.float32)
    c = np.zeros((2, 1, 64), dtype=np.float32)
    sr = np.array(SR, dtype=np.int64)
    n = len(audio) // VAD_CHUNK
    probs = []
    for i in range(n):
        chunk = audio[i*VAD_CHUNK:(i+1)*VAD_CHUNK].astype(np.float32)[np.newaxis, :]
        out, h, c = vad_sess.run(None, {"input": chunk, "h": h, "c": c, "sr": sr})
        probs.append(float(out[0, 0]))
    if not probs:
        return [(0.0, len(audio)/SR, audio)]
    MIN_CH, PAD, MERGE = 16, 5, 16
    raw, in_sp, s0 = [], False, 0
    for i, p in enumerate(probs):
        if p >= VAD_THRESHOLD and not in_sp:
            s0, in_sp = i, True
        elif p < VAD_THRESHOLD and in_sp:
            if i - s0 >= MIN_CH:
                raw.append((max(0, s0-PAD), min(n, i+PAD)))
            in_sp = False
    if in_sp and n - s0 >= MIN_CH:
        raw.append((max(0, s0-PAD), n))
    if not raw:
        return []
    merged = [list(raw[0])]
    for s, e in raw[1:]:
        if s - merged[-1][1] <= MERGE:
            merged[-1][1] = e
        else:
            merged.append([s, e])
    mx = MAX_GROUP_SEC * SR
    groups, gs = [], merged[0][0]*VAD_CHUNK
    ge = merged[0][1]*VAD_CHUNK
    for s_, e_ in merged[1:]:
        s, e = s_*VAD_CHUNK, e_*VAD_CHUNK
        if e - gs > mx:
            groups.append((gs, ge)); gs = s
        ge = e
    groups.append((gs, ge))
    res, tail = [], int(0.35 * SR)
    for gs, ge in groups:
        ch = audio[gs:min(len(audio), ge+tail)].astype(np.float32)
        if len(ch) < SR // 2:
            continue
        res.append((gs/SR, ge/SR, ch))
    return res

def srt_ts(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600); m, s = divmod(rem, 60)
    ms = int(round((sec - int(sec)) * 1000))
    if ms == 1000: s, ms = s + 1, 0
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def split_to_lines(text):
    """無 aligner 時的 fallback：標點斷句（標點不輸出）"""
    punct = _ZH_CLAUSE_END | _EN_SENT_END
    lines, buf = [], ''
    i = 0
    while i < len(text):
        ch = text[i]
        if ch in punct:
            if buf.strip(): lines.append(buf.strip())
            buf = ''; i += 1; continue
        if ch.isalpha() and ord(ch) < 128:
            j = i
            while j < len(text) and text[j].isalpha() and ord(text[j]) < 128: j += 1
            word = text[i:j]
            prefix = ' ' if buf and not buf.endswith(' ') else ''
            if len(buf) + len(prefix) + len(word) > MAX_CHARS and buf.strip():
                lines.append(buf.strip()); buf = word
            else:
                buf += prefix + word
            i = j; continue
        if ch == ' ':
            if buf and not buf.endswith(' '): buf += ' '
            i += 1
            if len(buf.rstrip()) >= MAX_CHARS:
                lines.append(buf.strip()); buf = ''
            continue
        buf += ch; i += 1
        if len(buf) >= MAX_CHARS:
            lines.append(buf.strip()); buf = ''
    if buf.strip(): lines.append(buf.strip())
    return lines

def words_to_lines(items, offset):
    """ForcedAligner 字級時間軸 → 字幕行（offset=chunk 起始秒數）"""
    punct = _ZH_CLAUSE_END | _EN_SENT_END
    lines, buf, s0 = [], [], None
    def flush(end):
        nonlocal buf, s0
        if buf:
            lines.append((s0 + offset, end + offset, ''.join(buf).strip()))
        buf, s0 = [], None
    for it in items:
        w = it.text
        if w in punct:  # 標點只當切點
            flush(it.end_time); continue
        if s0 is None: s0 = it.start_time
        sep = ' ' if (buf and w and ord(w[0]) < 128 and w[0].isalpha()) else ''
        if len(''.join(buf)) + len(sep) + len(w) > MAX_CHARS and buf:
            flush(it.start_time); s0 = it.start_time
        buf.append(sep + w)
        last_end = it.end_time
    flush(last_end if buf else 0)
    return [(s, e, t) for s, e, t in lines if t]

print('函式就緒')


In [ ]:
# ═══ 執行轉錄 ═══
import time
from tqdm.auto import tqdm

audio = load_audio_16k_mono(AUDIO_PATH)
print(f"音檔：{len(audio)/SR:.1f}s")
groups = detect_speech_groups(audio, vad_sess)
print(f"VAD 切出 {len(groups)} 段")
assert groups, 'VAD 沒有偵測到語音，請檢查音檔'

import opencc
cc = opencc.OpenCC('s2twp') if DO_OPENCC else None
def conv(t, lang=''):
    # 日語輸出保留原生漢字，不做簡轉繁
    if cc is None or 'japan' in (lang or '').lower():
        return t
    return cc.convert(t)

subs = []
t_all = time.time()
for g0, g1, chunk in tqdm(groups, desc='轉錄中'):
    r = model.transcribe([(chunk, SR)], language=LANGUAGE or None,
                         context=HINT, return_time_stamps=USE_ALIGNER)[0]
    text = (r.text or '').strip()
    if not text:
        continue
    rlang = getattr(r, 'language', '') or ''
    if USE_ALIGNER and r.time_stamps:
        for s, e, t in words_to_lines(r.time_stamps, g0):
            subs.append((s, e, conv(t, rlang)))
    else:  # fallback：按字數比例分配時間
        lines = split_to_lines(conv(text, rlang))
        weights = [max(1, len(l)) for l in lines] or [1]
        tot = sum(weights); cur = g0
        for l, w in zip(lines, weights):
            dur = (g1 - g0) * w / tot
            subs.append((cur, cur + dur, l)); cur += dur

print(f"✅ 完成：{len(subs)} 行字幕，耗時 {time.time()-t_all:.0f}s")

SRT_PATH = "/content/" + AUDIO_PATH.rsplit("/", 1)[-1].rsplit(".", 1)[0] + ".srt"
with open(SRT_PATH, "w", encoding="utf-8") as f:
    for i, (s, e, t) in enumerate(subs, 1):
        f.write(f"{i}\n{srt_ts(s)} --> {srt_ts(e)}\n{t}\n\n")
print("SRT：", SRT_PATH)
# 預覽前 10 行
for s, e, t in subs[:10]:
    print(f"[{srt_ts(s)}] {t}")


In [ ]:
# ═══ 下載 SRT ═══
from google.colab import files
files.download(SRT_PATH)


---
**備註**
- 換模型：改第一格的 `MODEL_ID`（例：日語動漫版），重跑「載入模型」那格之後的儲存格即可。
- `LANGUAGE=None` 為自動偵測；已知語言時指定會更準。
- 免費版 Colab 斷線後，依序重跑所有儲存格即可恢復（`Shift` 鍵全選或「執行階段 → 全部執行」）。
- 長音檔（>30 分鐘）建議先確認 VRAM 餘量；轉錄是逐段跑，記憶體主要花在模型常駐。